# Positional Encoding Toy Experiment

This notebook tests whether a **single attention layer** can learn to map a uniform token input to normalized $(x, y)$ coordinates.

We compare four positional encoding styles:
- Learned additive embeddings
- Fixed 2D sinusoidal additive embeddings
- RoPE (rotary) on $(q, k)$
- ALiBi attention bias

The key idea: if a positional method provides a useful positional bias, this toy objective should be easy to optimize.

In [1]:
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from dinosaw.utils import add_custom_font


def set_seed(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


In [2]:
def build_2d_sincos_pos_embed(
    h: int,
    w: int,
    embed_dim: int,
    dtype,
    device,
    base_wavelength: int = 10_000,
) -> torch.Tensor:
    """Returns shape (1, h*w, embed_dim)."""
    assert embed_dim % 4 == 0, "embed_dim must be divisible by 4"

    grid_h = torch.arange(h, dtype=dtype, device=device)
    grid_w = torch.arange(w, dtype=dtype, device=device)
    y, x = torch.meshgrid(grid_h, grid_w, indexing="ij")

    y = y.reshape(-1)
    x = x.reshape(-1)

    dim_quarter = embed_dim // 4
    omega = torch.arange(dim_quarter, dtype=dtype, device=device)
    omega = 1.0 / (base_wavelength ** (omega / dim_quarter))

    out_y = y[:, None] * omega[None, :]
    out_x = x[:, None] * omega[None, :]

    emb_y = torch.cat([torch.sin(out_y), torch.cos(out_y)], dim=1)
    emb_x = torch.cat([torch.sin(out_x), torch.cos(out_x)], dim=1)

    pos_embed = torch.cat([emb_y, emb_x], dim=1)
    return pos_embed.unsqueeze(0)


def get_distance_matrix(
    n_tokens_h: int,
    n_tokens_w: int,
    n_reg_tokens: int = 0,
    metric: str = "euclidean",
    normalize: bool = True,
    wrap: bool = False,
    add_cls: bool = False,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
) -> torch.Tensor:
    coords = torch.stack(
        torch.meshgrid(
            torch.arange(n_tokens_h, device=device),
            torch.arange(n_tokens_w, device=device),
            indexing="ij",
        ),
        dim=-1,
    ).reshape(-1, 2)

    diff = coords.unsqueeze(1) - coords.unsqueeze(0)
    diff = diff.abs()

    if wrap:
        diff[..., 0] = torch.minimum(diff[..., 0], n_tokens_h - diff[..., 0])
        diff[..., 1] = torch.minimum(diff[..., 1], n_tokens_w - diff[..., 1])

    if metric == "euclidean":
        d = torch.sqrt((diff**2).sum(-1))
    elif metric == "manhattan":
        d = diff.sum(-1)
    else:
        raise ValueError("metric must be 'euclidean' or 'manhattan'")

    if normalize:
        d = d / d.max().clamp_min(1e-8)
    d = -d

    n_extra_tokens = int(add_cls) + n_reg_tokens
    d = F.pad(d, (n_extra_tokens, 0, n_extra_tokens, 0), mode="constant")
    return d.to(device=device, dtype=dtype)


def build_normalized_xy_grid(h: int, w: int, device: torch.device, dtype: torch.dtype = torch.float32):
    y = torch.linspace(0.0, 1.0, h, device=device, dtype=dtype)
    x = torch.linspace(0.0, 1.0, w, device=device, dtype=dtype)
    yy, xx = torch.meshgrid(y, x, indexing="ij")
    grid = torch.stack([xx, yy], dim=0).unsqueeze(0)  # (1, 2, H, W)
    return grid


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    x_rot = torch.stack((-x2, x1), dim=-1)
    return x_rot.flatten(-2)


def build_1d_rope_trig(
    positions: torch.Tensor,
    dim: int,
    base_wavelength: int = 10_000,
):
    """Returns cos/sin for one axis with shape (N, dim)."""
    assert dim % 2 == 0, "RoPE axis dim must be even"
    half = dim // 2
    omega = torch.arange(half, dtype=positions.dtype, device=positions.device)
    omega = 1.0 / (base_wavelength ** (omega / half))
    theta = positions[:, None] * omega[None, :]
    theta = torch.repeat_interleave(theta, repeats=2, dim=-1)
    return torch.cos(theta), torch.sin(theta)


def build_2d_rope_cache(
    h: int,
    w: int,
    head_dim: int,
    dtype: torch.dtype,
    device: torch.device,
    base_wavelength: int = 10_000,
):
    """
    Axial 2D RoPE cache.

    Splits head_dim into two halves: first half rotates with y positions,
    second half rotates with x positions.

    Returns cos, sin of shape (1, 1, H*W, head_dim).
    """
    assert head_dim % 4 == 0, "For axial 2D RoPE, head_dim must be divisible by 4"

    axis_dim = head_dim // 2

    grid_h = torch.arange(h, dtype=dtype, device=device)
    grid_w = torch.arange(w, dtype=dtype, device=device)
    yy, xx = torch.meshgrid(grid_h, grid_w, indexing="ij")
    yy = yy.reshape(-1)
    xx = xx.reshape(-1)

    y_cos, y_sin = build_1d_rope_trig(yy, axis_dim, base_wavelength=base_wavelength)
    x_cos, x_sin = build_1d_rope_trig(xx, axis_dim, base_wavelength=base_wavelength)

    cos = torch.cat([y_cos, x_cos], dim=-1).unsqueeze(0).unsqueeze(0)
    sin = torch.cat([y_sin, x_sin], dim=-1).unsqueeze(0).unsqueeze(0)
    return cos, sin


def apply_rope_2d(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    return q, k


def build_1D_rope_cache(
    h: int,
    w: int,
    head_dim: int,
    dtype: torch.dtype,
    device: torch.device,
    base_wavelength: int = 10_000,
):
    """
    1D RoPE cache for flattened 2D tokens.

    Returns cos, sin of shape (1, 1, H*W, head_dim).
    """
    assert head_dim % 2 == 0, "For 1D RoPE, head_dim must be divisible by 2"

    grid_h = torch.arange(h, dtype=dtype, device=device)
    grid_w = torch.arange(w, dtype=dtype, device=device)
    yy, xx = torch.meshgrid(grid_h, grid_w, indexing="ij")
    positions = (yy * w + xx).reshape(-1)

    cos, sin = build_1d_rope_trig(positions, head_dim, base_wavelength=base_wavelength)
    return cos.unsqueeze(0).unsqueeze(0), sin.unsqueeze(0).unsqueeze(0)

# def apply_rope_1d(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
#     q = q * cos + rotate_half(q) * sin
#     k = k * cos + rotate_half(k) * sin
#     return q, k


def get_alibi_slopes(num_heads: int, device: torch.device, mode: str = "fixed") -> torch.Tensor:
    if mode == "constant":
        slopes = torch.ones(num_heads, device=device)
    elif mode == "fixed":
        xs = (2**8) ** (1 / num_heads)
        slopes = torch.tensor([1 / xs ** (i + 1) for i in range(num_heads)], device=device)
    else:
        raise ValueError("mode must be 'fixed' or 'constant'")
    return slopes.view(num_heads, 1, 1)


def add_positional_encoding(x: torch.Tensor, pos: torch.Tensor | None):
    if pos is None:
        return x
    return x + pos

In [3]:
@dataclass
class ToyConfig:
    h: int = 14
    w: int = 14
    dim: int = 32
    num_heads: int = 6
    batch_size: int = 32
    base_wavelength: int = 10_000
    lr: float = 1e-3
    steps: int = 800
    log_every: int = 100


def gen_sample_mask(
    shape: tuple[int, int],
    step: int = 4,
    cutoff_frac: float = 1.0,
    random_mask: bool = False,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Generate a spatial sampling mask over a top-left region with optional random sampling."""
    h, w = shape
    if step < 1:
        raise ValueError("step must be >= 1")
    if not (0.0 < cutoff_frac <= 1.0):
        raise ValueError("cutoff_frac must be in (0, 1]")

    mask = torch.zeros((h, w), dtype=torch.bool, device=device)
    y1 = max(1, int(h * cutoff_frac))
    x1 = max(1, int(w * cutoff_frac))

    if random_mask:
        n_total = y1 * x1
        n_samples = max(1, n_total // (step * step))
        inds = torch.randperm(n_total, device=device)[:n_samples]
        flat_mask = torch.zeros(n_total, dtype=torch.bool, device=device)
        flat_mask[inds] = True
        mask[:y1, :x1] = flat_mask.reshape(y1, x1)
    else:
        mask[:y1:step, :x1:step] = True

    return mask


class CustomAttentionBlock(nn.Module):
    """
    Single self-attention block configurable with one positional mode:
    - learned: additive learned token embedding
    - sincos: additive fixed 2D sinusoidal embedding
    - rope: rotary embedding on q,k
    - alibi: additive attention bias from pairwise distances
    """

    def __init__(self, cfg: ToyConfig, pos_mode: str, base_wavelength: int = 10_000):
        super().__init__()
        assert pos_mode in {"learned", "sincos", "rope_1D", "rope_2D", "alibi", "nope"}
        assert cfg.dim % cfg.num_heads == 0

        self.cfg = cfg
        self.pos_mode = pos_mode
        self.n = cfg.h * cfg.w
        self.head_dim = cfg.dim // cfg.num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(cfg.dim, 3 * cfg.dim)
        self.out_proj = nn.Linear(cfg.dim, 2)

        if pos_mode == "learned":
            self.pos_embed = nn.Parameter(torch.zeros(1, self.n, cfg.dim))
            nn.init.trunc_normal_(self.pos_embed, std=0.2)
        else:
            self.pos_embed = None

        if pos_mode == "sincos":
            pe = build_2d_sincos_pos_embed(
                cfg.h,
                cfg.w,
                cfg.dim,
                dtype=torch.float32,
                device=torch.device("cpu"),
                base_wavelength=base_wavelength,
            )
            self.register_buffer("sincos_embed", pe, persistent=False)
        else:
            self.sincos_embed = None

        if pos_mode == "rope_2D":
            cos, sin = build_2d_rope_cache(
                cfg.h,
                cfg.w,
                self.head_dim,
                dtype=torch.float32,
                device=torch.device("cpu"),
                base_wavelength=base_wavelength,
            )
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        elif pos_mode == "rope_1D":
            cos, sin = build_1D_rope_cache(
                cfg.h,
                cfg.w,
                self.head_dim,
                dtype=torch.float32,
                device=torch.device("cpu"),
                base_wavelength=base_wavelength,
            )
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        else:
            self.rope_cos = None
            self.rope_sin = None

        if pos_mode == "alibi":
            d = get_distance_matrix(
                cfg.h,
                cfg.w,
                n_reg_tokens=0,
                add_cls=False,
                metric="euclidean",
                wrap=False,
                normalize=True,
                device="cpu",
            )
            self.register_buffer("alibi_distance", d, persistent=False)
            m = get_alibi_slopes(cfg.num_heads, device=torch.device("cpu"), mode="fixed")
            self.register_buffer("alibi_slopes", m, persistent=False)
        else:
            self.alibi_distance = None
            self.alibi_slopes = None

    def _move_runtime_buffers(self, x: torch.Tensor):
        if self.sincos_embed is not None and self.sincos_embed.device != x.device:
            self.sincos_embed = self.sincos_embed.to(device=x.device, dtype=x.dtype)
        if self.rope_cos is not None and self.rope_cos.device != x.device:
            self.rope_cos = self.rope_cos.to(device=x.device, dtype=x.dtype)
            self.rope_sin = self.rope_sin.to(device=x.device, dtype=x.dtype)
        if self.alibi_distance is not None and self.alibi_distance.device != x.device:
            self.alibi_distance = self.alibi_distance.to(device=x.device, dtype=x.dtype)
        if self.alibi_slopes is not None and self.alibi_slopes.device != x.device:
            self.alibi_slopes = self.alibi_slopes.to(device=x.device, dtype=x.dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, n, c = x.shape
        assert n == self.n
        self._move_runtime_buffers(x)

        if self.pos_mode == "learned":
            x = add_positional_encoding(x, self.pos_embed)
        elif self.pos_mode == "sincos":
            x = add_positional_encoding(x, self.sincos_embed)

        qkv = self.qkv(x).view(b, n, 3, self.cfg.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        if "rope" in self.pos_mode :
            q, k = apply_rope_2d(q, k, self.rope_cos, self.rope_sin)

        attn = (q * self.scale) @ k.transpose(-2, -1)

        if self.pos_mode == "alibi":
            bias = (self.alibi_slopes * self.alibi_distance).unsqueeze(0)
            attn = attn + bias

        attn = attn.softmax(dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).reshape(b, n, c)
        out = self.out_proj(out)
        return out


class ToyCoordinatePredictor(nn.Module):
    def __init__(self, cfg: ToyConfig, pos_mode: str):
        super().__init__()
        self.cfg = cfg
        self.input_proj = nn.Linear(1, cfg.dim)
        self.attn = CustomAttentionBlock(cfg, pos_mode=pos_mode)
        self.head = nn.Sequential(nn.Identity())

    def forward(self, x_uniform_tokens: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x_uniform_tokens)
        x = self.attn(x)
        pred = self.head(x)
        return pred

def to_numpy(t: torch.Tensor) -> np.ndarray:
    return t.detach().cpu().numpy()

def train_one_mode(pos_mode: str, cfg: ToyConfig, device: torch.device):
    model = ToyCoordinatePredictor(cfg, pos_mode=pos_mode).to(device)
    model.train()

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.0)

    grid = build_normalized_xy_grid(cfg.h, cfg.w, device=device)
    target_tokens = grid.flatten(2).transpose(1, 2)

    losses = []
    x_in = torch.rand(cfg.batch_size, cfg.h * cfg.w, 1, device=device)
    # x_in = torch.zeros(cfg.batch_size, cfg.h * cfg.w, 1, device=device)
    # print(x_in.shape)
    # x_in[:, 0, :] = 1
    y = target_tokens.expand(cfg.batch_size, -1, -1)

    for step in range(1, cfg.steps + 1):
        pred = model(x_in)

        loss = F.l1_loss(pred, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        losses.append(float(loss.item()))
        if step % cfg.log_every == 0:
            print(f"[{pos_mode:7s}] step={step:4d} loss={loss.item():.6f}")

    model.eval()
    with torch.no_grad():
        pred_eval = model(x_in)

    pred_grid = pred_eval.transpose(1, 2).reshape(1, 2, cfg.h, cfg.w)
    return {
        "mode": pos_mode,
        # "model": model,
        "losses": losses,
        "final_loss": losses[-1],
        "input_grid": to_numpy(x_in[0, :, 0].reshape(cfg.h, cfg.w)),
        "target_grid": to_numpy(grid),
        "pred_grid": to_numpy(pred_grid),
    }, model

In [4]:
# interesting:
# hot pixel, 32x32, d=3, h=4, lr=5e-4, steps=2000, log_every=100
# alibi is radial fall off, 2d rope is perfect, 1d rope uses mix of short and long rotations to ape grid structure

cfg = ToyConfig(h=32, w=32, dim=32, num_heads=4, batch_size=1, lr=5e-4, steps=5_000, log_every=100)
results = {}
models = {}

# for mode in ["learned", "sincos", "rope", "alibi"]:
for mode in ["learned", "sincos", "rope_1D", "rope_2D", "alibi", "nope"]:
    print(f"\nTraining mode: {mode}")
    res, model = train_one_mode(mode, cfg, device=device)
    results[mode] = res
    models[mode] = model

print("\nFinal losses:")
for mode, res in results.items():
    print(f"{mode:7s}: {res['final_loss']:.6f}")


Training mode: learned
[learned] step= 100 loss=0.257949
[learned] step= 200 loss=0.254985
[learned] step= 300 loss=0.155713
[learned] step= 400 loss=0.131897
[learned] step= 500 loss=0.120400
[learned] step= 600 loss=0.061275
[learned] step= 700 loss=0.016603
[learned] step= 800 loss=0.009813
[learned] step= 900 loss=0.007523
[learned] step=1000 loss=0.003308
[learned] step=1100 loss=0.006725
[learned] step=1200 loss=0.004295
[learned] step=1300 loss=0.005120
[learned] step=1400 loss=0.005164
[learned] step=1500 loss=0.004593
[learned] step=1600 loss=0.002367
[learned] step=1700 loss=0.003164
[learned] step=1800 loss=0.005055
[learned] step=1900 loss=0.004259
[learned] step=2000 loss=0.003575
[learned] step=2100 loss=0.004858
[learned] step=2200 loss=0.004331
[learned] step=2300 loss=0.003795
[learned] step=2400 loss=0.004488
[learned] step=2500 loss=0.004899
[learned] step=2600 loss=0.004478
[learned] step=2700 loss=0.003588
[learned] step=2800 loss=0.003776
[learned] step=2900 loss

In [5]:

# results = pload(open("data/6/th_toy_model_results.pkl", "rb"))

# plt.style.use("thesis.mplstyle")
# 2x7 summary figure with a spanning loss panel

n_rows, n_cols = 2, 8


w_ratios = [1, 0.2, 1, 1, 1, 1, 1, 1]
# fig = plt.figure(figsize=(W * sum(w_ratios), H * n_rows))


W, H = 7.5, 1  * 2.3
fig = plt.figure(figsize=(W, H))

plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False

add_custom_font('resources/fonts', 'Grotesk')

gs = fig.add_gridspec(n_rows, n_cols, wspace=0.4, width_ratios=w_ratios)

# import matplotlib as mpl
# mpl.rcParams['figure.constrained_layout.w_pad'] = 0.02
# mpl.rcParams['figure.constrained_layout.h_pad'] = 0.02
# mpl.rcParams['figure.constrained_layout.wspace'] = 0.02
# mpl.rcParams['figure.constrained_layout.hspace'] = 0.02

# fig.set_constrained_layout_pads(
#     w_pad=0.02,
#     h_pad=0.02,
#     wspace=0.02,
#     hspace=0.02
# )

titles = {"learned": "Learned", "sincos": "Sinusoidal", "rope_1D": "RoPE (1D)", "rope_2D": "RoPE (2D)", "alibi": "ALiBi", "nope": "NoPE"}

# Loss plot spans first 2 rows and first 2 columns
loss_ax = fig.add_subplot(gs[:, 2:5])
for mode, res in results.items():
    loss_ax.plot(res["losses"], label=titles[mode])
loss_ax.set_yscale("log")
loss_ax.set_xlabel("Step", )
loss_ax.set_ylabel("L1 loss", )
loss_ax.set_title("Training loss of attention layer", )
loss_ax.tick_params(axis='both', labelsize=6)
loss_ax.grid(alpha=0.2)
loss_ax.legend(fontsize=5, ncol=2, loc='lower left')


def xy_to_rb_rgb(xy_map: np.ndarray) -> np.ndarray:
    if xy_map.ndim != 3 or xy_map.shape[0] != 2:
        raise ValueError("xy_map must have shape (2, H, W)")
    x = xy_map[0].clip(0.0, 1.0)
    y = xy_map[1].clip(0.0, 1.0)
    g = np.zeros_like(x)
    return np.stack([x, g, y], axis=-1)

input_grid = results["sincos"]["input_grid"]
target_rgb = xy_to_rb_rgb(results["sincos"]["target_grid"][0])

def hide_axes(ax, set_lims: bool = True):
    ax.set_xticks([])
    ax.set_yticks([])

    if set_lims:
        ax.set_xlim(-0.5, input_grid.shape[1]-0.5)
        ax.set_ylim(input_grid.shape[0]-0.5, -0.5)


TITLE_PAD = 14.5
ax_input = fig.add_subplot(gs[0, 0])
ax_input.imshow(input_grid, cmap="gray", interpolation="nearest")
ax_input.set_title("Random input",  pad=TITLE_PAD)
hide_axes(ax_input)

ax_target = fig.add_subplot(gs[1, 0])
ax_target.imshow(target_rgb)
ax_target.set_title("Target ramp", )
ax_target.set_xlabel("R = x", )
ax_target.set_ylabel("B = y", )
hide_axes(ax_target)

ax_sinusoid = fig.add_subplot(gs[0, 5])
sinusoid_rgb = xy_to_rb_rgb(results["sincos"]["pred_grid"][0])
ax_sinusoid.imshow(sinusoid_rgb)
ax_sinusoid.set_xlabel("Sinusoid", )
hide_axes(ax_sinusoid)

ax_learned = fig.add_subplot(gs[0, 6])
learned_rgb = xy_to_rb_rgb(results["learned"]["pred_grid"][0])
ax_learned.imshow(learned_rgb)
ax_learned.set_xlabel("Learned", )
ax_learned.set_title("Predictions at end of training",  pad=TITLE_PAD)
hide_axes(ax_learned)

ax_rope_1D = fig.add_subplot(gs[0, 7])
rope_1D_rgb = xy_to_rb_rgb(results["rope_1D"]["pred_grid"][0])
ax_rope_1D.imshow(rope_1D_rgb)
ax_rope_1D.set_xlabel("RoPE (1D)", )
hide_axes(ax_rope_1D)


for i, rel_enc in enumerate(["rope_2D", "alibi", "nope"]):
    if rel_enc in results:
        ax = fig.add_subplot(gs[1, 5 + i])
        pred_rgb = xy_to_rb_rgb(results[rel_enc]["pred_grid"][0])
        ax.imshow(pred_rgb)
        ax.set_xlabel(titles[rel_enc], )
        hide_axes(ax)
        # ax.tick_params(pad=0)


SAVE = True
if SAVE:
    plt.savefig("saved/04.pdf", dpi=300, bbox_inches='tight')
    plt.close()
    # plt.savefig("../6_homog/toy_model.pdf", bbox_inches="tight")
# plt.close()

findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
